# Lezione 55 — Embedding, retrieval e grafo

> **Modulo:** capstone · **Tempo stimato:** 30 minuti
> **Prerequisiti:** Lezioni 18 (coseno), 26 (entita'), 27 (grafo).
>
> Obiettivo pratico unico: assemblare i componenti di **rappresentazione** —
> embedding + ricerca per similarita' + estrazione entita' + grafo — in NumPy e
> Python puro (senza dipendenze pesanti).

## Teoria essenziale

Tre pezzi del record vengono da qui: `embedding` (per la ricerca), `entities`
(chi/cosa compare) e `relations` (il grafo che le collega). Riusiamo:
l'embedding deterministico delle Lezioni 30-31, la similarita' coseno della
Lezione 18 e l'estrazione a regole della Lezione 26. Il grafo lo teniamo come
**lista di adiacenza** (Python puro, niente `networkx`), cosi' il capstone gira
ovunque.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

train = pd.read_csv(Path('..') / 'datasets' / 'processed' / 'memory_train.csv')

DIM = 48
def embed(testo):
    v = np.zeros(DIM)
    for w in re.findall(r"[a-zA-Z]+", str(testo).lower()):
        v[int.from_bytes(w.encode(), 'little') % DIM] += 1.0
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

def cosine(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return float(a @ b / (na * nb)) if na and nb else 0.0

banca = train.head(30).reset_index(drop=True)
E = np.vstack([embed(t) for t in banca['text']])
print("indice di embedding:", E.shape)

indice di embedding: (30, 48)


In [2]:
def cerca(query, k=3):
    q = embed(query)
    sim = E @ q                     # coseno (E gia' normalizzato, q pure)
    top = np.argsort(sim)[::-1][:k]
    return [(float(sim[i]), banca['text'].iloc[i]) for i in top]

for peso, testo in cerca("The user prefers morning meetings", k=3):
    print(f"  {peso:.3f}  {testo}")

  0.674  The user prefers morning sessions for il progetto TensorFlow.
  0.600  The user likes walking meetings.
  0.600  The user likes walking meetings.


In [3]:
STOP = {"The", "A", "User"}
def estrai_entita(testo):
    cand = re.findall(r"\b[A-Z][a-zA-Z]+\b", str(testo))
    return [c for c in cand if c not in STOP]

# grafo come lista di adiacenza: memoria -> entita' (bipartito)
def costruisci_grafo(df):
    adj = {}
    for _, riga in df.iterrows():
        mid = riga['memory_id']
        for e in estrai_entita(riga['text']):
            adj.setdefault(mid, set()).add(e)
            adj.setdefault(e, set()).add(mid)      # non orientato
    return adj

grafo = costruisci_grafo(banca)
# entita' piu' connesse
entita = [(n, len(v)) for n, v in grafo.items() if n[0].isupper() and not n.startswith('hist')]
entita.sort(key=lambda x: -x[1])
print("entita' piu' connesse:", entita[:5])

entita' piu' connesse: [('Elena', 5), ('Lucia', 4), ('Glasgow', 4), ('Bologna', 3), ('Milano', 3)]


Leggi i risultati: la ricerca restituisce le memorie piu' simili alla query, e
il grafo collega memorie ed entita'. Le entita' piu' connesse sono i "nodi
centrali" della conoscenza accumulata.

In [4]:
# il componente combinato per la pipeline
def rappresenta(testo):
    return {"embedding_dim": DIM, "entities": estrai_entita(testo)}

r = rappresenta("Marco visited Glasgow with his son.")
# controlli di non-regressione
assert r["embedding_dim"] == DIM
assert "Marco" in r["entities"] and "Glasgow" in r["entities"]
assert cerca("morning", k=2) and all(0 <= s <= 1.0001 for s, _ in cerca("morning", k=2))
print("componente rappresentazione OK:", r)

componente rappresentazione OK: {'embedding_dim': 48, 'entities': ['Marco', 'Glasgow']}


## Riepilogo (max 8 punti)

1. Da qui vengono `embedding`, `entities` e `relations`.
2. Embedding deterministico (Lezioni 30-31), niente modello esterno.
3. Ricerca per **similarita' coseno** (Lezione 18).
4. Estrazione entita' a regole (Lezione 26).
5. Grafo come **lista di adiacenza** in Python puro (niente `networkx`).
6. Le entita' piu' connesse sono i nodi centrali.
7. `rappresenta(testo)` e' il componente per la pipeline (passo 55).
8. Tutto gira nell'ambiente di base, senza dipendenze pesanti.

## Quiz

1. Perche' l'embedding normalizzato semplifica il calcolo del coseno?
2. Cosa rappresenta un nodo con grado alto nel grafo?
3. Perche' usare una lista di adiacenza invece di `networkx` qui?

*(Risposte: 1. il coseno diventa un semplice prodotto scalare; 2. un'entita'
citata in molte memorie, centrale nella conoscenza; 3. per mantenere il capstone
eseguibile senza dipendenze opzionali.)*